# HW3
## Load Data


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score




In [3]:
data = pd.read_csv("04-evaluation/course_lead_scoring_2026.csv")
data.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Q1

In [6]:
data.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [7]:
numerical_features = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']
categorical_features = ['lead_source','industry','employment_status','location']

data[categorical_features] = data[categorical_features].fillna('NA')
data[numerical_features] = data[numerical_features].fillna(0)

In [9]:
df_full_train, df_test = train_test_split(data, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [10]:
y_train = df_train.converted.values
y_val = df_val.converted.values 
y_test = df_test.converted.values

In [17]:
for fea in ["lead_score", "number_of_courses_viewed", "interaction_count", "annual_income"]:
    auc = roc_auc_score(y_train, df_train[fea])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[fea])

    print(f"{fea}: {auc:.4f}")

lead_score: 0.7882
number_of_courses_viewed: 0.7230
interaction_count: 0.7649
annual_income: 0.6082


## Q2

In [19]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)

In [20]:
dicts_train = df_train[categorical_features + numerical_features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(dicts_train)

In [ ]:
dicts_val = df_val[categorical_features + numerical_features].to_dict(orient='records')
X_val = dv.transform(dicts_val)

In [23]:
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]

roc_auc_score(y_val,y_pred)
# sign_decision = (y_pred >= 0.5)
# round((sign_decision == y_val).mean(),2)

0.7318222288173319